In [7]:
import os
from dotenv import  load_dotenv
load_dotenv()

True

In [9]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
llm = ChatGroq(model='openai/gpt-oss-120b',temperature=0)

In [14]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
loader = CSVLoader(
            file_path='/Users/suriyaa/Documents/Projects/LLMOps/Anime-Recommender/data/anime_updated.csv',
            encoding='utf-8',
            metadata_columns=[]
        )
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(data)

persist_dir = "Chromadb"
db = Chroma.from_documents(texts, embedding, persist_directory=persist_dir)
db.persist()

In [16]:
vector_store = Chroma(persist_directory=persist_dir,embedding_function=embedding)
retriever = vector_store.as_retriever()

In [17]:
from langchain_core.prompts import PromptTemplate


def get_anime_prompt():
    template = """
You are an expert anime recommender. Your job is to help users find the perfect anime based on their preferences.

Using the following context, provide a detailed and engaging response to the user's question.

For each question, suggest exactly three anime titles. For each recommendation, include:
1. The anime title.
2. A concise plot summary (2-3 sentences).
3. A clear explanation of why this anime matches the user's preferences.

Present your recommendations in a numbered list format for easy reading.

If you don't know the answer, respond honestly by saying you don't know — do not fabricate any information.

Context:
{context}

User's question:
{question}

Your well-structured response:
"""

    return PromptTemplate(template=template, input_variables=["context", "question"])

In [21]:
prompt = get_anime_prompt()

In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# LCEL chain
rag_chain = (
    {
        "context": retriever,   # retriever auto-fills context
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
rag_chain.invoke('Demon Hunter')

'1. **Demon Slayer: Kimetsu no Yaiba**  \n   *Plot:* After his family is slaughtered by demons and his sister Nezuko is turned into one, teenage Tanjiro Kamado joins the Demon Slayer Corps to hunt the monsters and find a cure for his sister. He trains rigorously, mastering the Water Breathing style while confronting increasingly powerful demons.  \n   *Why it fits:* The series centers on a young protagonist who becomes a demon hunter, blending intense action, supernatural horror, and heartfelt drama—exactly the “demon hunter” vibe you’re looking for.\n\n2. **Jujutsu Kaisen**  \n   *Plot:* Yuji Itadori, a high‑schooler with extraordinary physical abilities, swallows a cursed finger to save his friends and becomes the host of the powerful Curse King Ryomen Sukuna. He enrolls in Tokyo Jujutsu High, where he learns to exorcise curses (demonic spirits) alongside fellow sorcerers.  \n   *Why it fits:* It follows a modern‑day demon‑hunting team battling cursed beings, offering fast‑paced batt